# Comparison of Two Networks

In [147]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pypsa

In [148]:

home = "/home/lucakristin/Desktop/my_pypsa"
cluster = "450"
analysis_dir = ""
log_file = ""

### Paths

### 1. Helper functions

Reusable utilities for loading solved networks and normalizing PyPSA outputs across different model variants.

In [149]:
from pathlib import Path


# --- Helper utilities -----------------------------------------------------


def _as_dataframe(obj: object) -> pd.DataFrame:
    """Return a DataFrame or an empty DataFrame for missing tables."""
    if obj is None:
        return pd.DataFrame()
    if isinstance(obj, pd.DataFrame):
        return obj
    return pd.DataFrame(obj)


def _empty_time_series(index: pd.Index, columns: pd.Index) -> pd.DataFrame:
    """Build a zero-filled time series with the given shape."""
    return pd.DataFrame(0.0, index=index, columns=columns)


def _safe_column(df: pd.DataFrame, *candidate_columns: str, default: str | None = None) -> str | None:
    """Return the first available column from a list of candidates."""
    for column in candidate_columns:
        if column in df.columns:
            return column
    return default


def _snapshot_weights(network: pypsa.Network, preferred: tuple[str, ...] = ("generators", "objective")) -> pd.Series:
    """Return snapshot weights aligned to the network snapshots."""
    weighting_table = getattr(network, "snapshot_weightings", None)
    if weighting_table is None:
        return pd.Series(1.0, index=network.snapshots)
    if isinstance(weighting_table, pd.DataFrame):
        for column in preferred:
            if column in weighting_table.columns:
                weights = weighting_table[column]
                break
        else:
            weights = weighting_table.iloc[:, 0] if not weighting_table.empty else pd.Series(1.0, index=network.snapshots)
    else:
        weights = pd.Series(weighting_table, index=network.snapshots)
    return pd.Series(weights, index=network.snapshots, dtype=float).reindex(network.snapshots).fillna(1.0)


def _weighted_sum(series_or_df: pd.DataFrame | pd.Series, network: pypsa.Network) -> float:
    """Compute a weighted sum over snapshots using the network time weighting."""
    weights = _snapshot_weights(network)
    if isinstance(series_or_df, pd.Series):
        aligned = series_or_df.reindex(network.snapshots).fillna(0.0)
        return float(aligned.mul(weights).sum())
    aligned = series_or_df.reindex(network.snapshots).fillna(0.0)
    return float(aligned.mul(weights, axis=0).sum().sum())


def _weighted_mean(series_or_df: pd.DataFrame | pd.Series, network: pypsa.Network) -> float:
    """Compute a weighted mean over snapshots."""
    weights = _snapshot_weights(network)
    if isinstance(series_or_df, pd.Series):
        aligned = series_or_df.reindex(network.snapshots).fillna(0.0)
        denominator = float(weights.sum())
        return float(aligned.mul(weights).sum() / denominator) if denominator else 0.0
    aligned = series_or_df.reindex(network.snapshots).fillna(0.0)
    denominator = float(weights.sum() * max(aligned.shape[1], 1))
    return float(aligned.mul(weights, axis=0).sum().sum() / denominator) if denominator else 0.0


def _weighted_bus_snapshot_mean(price_df: pd.DataFrame, network: pypsa.Network) -> float:
    """Average a bus x snapshot table over both axes using snapshot weights."""
    if price_df.empty:
        return 0.0
    weights = _snapshot_weights(network).reindex(price_df.index).fillna(1.0)
    denominator = float(weights.sum() * price_df.shape[1])
    return float(price_df.fillna(0.0).mul(weights, axis=0).sum().sum() / denominator) if denominator else 0.0


def _max_time_series_value(table: pd.DataFrame | pd.Series) -> float:
    """Return the maximum finite value in a time series table."""
    if table is None or getattr(table, "empty", True):
        return 0.0
    return float(np.nanmax(np.asarray(table, dtype=float)))


def _network_label(name: str) -> dict[str, str]:
    """Infer scenario, wind condition, and transmission setting from a run name."""
    wind_conditions = ("windvariability", "notwindy", "windy")
    wind_condition = next((condition for condition in wind_conditions if name.endswith(f"_{condition}")), "unknown")
    scenario = name.rsplit("_", 1)[0] if "_" in name else name
    transmission_setting = scenario.replace("germany_", "", 1) if scenario.startswith("germany_") else scenario
    return {
        "scenario": scenario,
        "wind_condition": wind_condition,
        "transmission_setting": transmission_setting,
    }


def _resolve_networks(network_inputs: dict[str, str | pypsa.Network]) -> dict[str, pypsa.Network]:
    """Load solved networks from disk or pass through existing PyPSA objects."""
    resolved_networks: dict[str, pypsa.Network] = {}
    for run_name, network_input in network_inputs.items():
        if isinstance(network_input, pypsa.Network):
            network = network_input
        else:
            network = pypsa.Network(str(network_input))
        network.name = run_name
        try:
            network._comparison_metadata = _network_label(run_name)
        except Exception:
            pass
        resolved_networks[run_name] = network
    return resolved_networks


def _line_capacity_series(network: pypsa.Network) -> pd.Series:
    """Return the effective line rating, preferring optimized capacity when available."""
    lines = _as_dataframe(getattr(network, "lines", None))
    if lines.empty:
        return pd.Series(dtype=float)
    if "s_nom_opt" in lines.columns:
        return lines["s_nom_opt"].fillna(lines.get("s_nom", 0.0)).astype(float)
    return lines.get("s_nom", pd.Series(index=lines.index, dtype=float)).astype(float)


def _generator_capacity_series(network: pypsa.Network) -> pd.Series:
    """Return the effective generator capacity, preferring optimized capacity when available."""
    generators = _as_dataframe(getattr(network, "generators", None))
    if generators.empty:
        return pd.Series(dtype=float)
    if "p_nom_opt" in generators.columns:
        return generators["p_nom_opt"].fillna(generators.get("p_nom", 0.0)).astype(float)
    return generators.get("p_nom", pd.Series(index=generators.index, dtype=float)).astype(float)


def _line_flow_series(network: pypsa.Network) -> pd.DataFrame:
    """Return the absolute line flow table used for loading calculations."""
    lines_t = getattr(network, "lines_t", None)
    if lines_t is None:
        return pd.DataFrame()
    if hasattr(lines_t, "p0") and not getattr(lines_t.p0, "empty", True):
        return lines_t.p0.abs()
    if hasattr(lines_t, "p1") and not getattr(lines_t.p1, "empty", True):
        return lines_t.p1.abs()
    return pd.DataFrame(index=network.snapshots, columns=getattr(network, "lines", pd.DataFrame()).index, dtype=float)


def _line_loading_table(network: pypsa.Network) -> pd.DataFrame:
    """Return line loading in percent for each snapshot and line."""
    capacities = _line_capacity_series(network).replace(0.0, np.nan)
    flows = _line_flow_series(network)
    if flows.empty or capacities.empty:
        return pd.DataFrame(index=network.snapshots, columns=getattr(network, "lines", pd.DataFrame()).index, dtype=float)
    loading = flows.reindex(columns=capacities.index).divide(capacities, axis=1) * 100.0
    return loading.replace([np.inf, -np.inf], np.nan)


def _load_shedding_generators(network: pypsa.Network) -> pd.Index:
    """Find generator columns that represent load shedding."""
    generators = _as_dataframe(getattr(network, "generators", None))
    if generators.empty or "carrier" not in generators.columns:
        return pd.Index([])
    carrier = generators["carrier"].fillna("").astype(str).str.lower()
    mask = carrier.str.contains("load") | carrier.str.contains("shed")
    return generators.index[mask]


def load_shedding_by_bus(network: pypsa.Network) -> pd.DataFrame:
    """Return load shedding at bus level with snapshots on the index."""
    shedding_generators = _load_shedding_generators(network)
    buses = getattr(network, "buses", pd.DataFrame()).index
    if len(shedding_generators) == 0:
        return _empty_time_series(network.snapshots, buses)
    generators = network.generators.loc[shedding_generators]
    time_series = getattr(network.generators_t, "p", pd.DataFrame()).reindex(columns=shedding_generators).fillna(0.0)
    if time_series.empty:
        return _empty_time_series(network.snapshots, buses)
    bus_series = generators["bus"].fillna("unknown") if "bus" in generators.columns else pd.Series("unknown", index=generators.index)
    return (
        time_series.T.groupby(bus_series).sum().T.reindex(index=network.snapshots, columns=buses, fill_value=0.0)
    )


def load_shedding_by_snapshot(network: pypsa.Network) -> pd.Series:
    """Return total load shedding by snapshot."""
    shedding = load_shedding_by_bus(network)
    if shedding.empty:
        return pd.Series(0.0, index=network.snapshots)
    return shedding.sum(axis=1).reindex(network.snapshots).fillna(0.0)


def carrier_dispatch_table(network: pypsa.Network) -> pd.DataFrame:
    """Aggregate generator dispatch by carrier for each snapshot."""
    generators = _as_dataframe(getattr(network, "generators", None))
    dispatch = getattr(network.generators_t, "p", pd.DataFrame())
    if generators.empty or dispatch.empty:
        return pd.DataFrame(index=network.snapshots)
    return dispatch.reindex(columns=generators.index, fill_value=0.0).T.groupby(generators["carrier"].fillna("unknown")).sum().T


def carrier_available_table(network: pypsa.Network) -> pd.DataFrame:
    """Aggregate available generator energy by carrier for each snapshot."""
    generators = _as_dataframe(getattr(network, "generators", None))
    if generators.empty:
        return pd.DataFrame(index=network.snapshots)
    p_max_pu = getattr(network.generators_t, "p_max_pu", pd.DataFrame())
    if p_max_pu.empty:
        p_max_pu = pd.DataFrame(1.0, index=network.snapshots, columns=generators.index)
    p_max_pu = p_max_pu.reindex(index=network.snapshots, columns=generators.index, fill_value=0.0)
    capacities = _generator_capacity_series(network).reindex(generators.index).fillna(0.0)
    available = p_max_pu.multiply(capacities, axis=1)
    return available.T.groupby(generators["carrier"].fillna("unknown")).sum().T


def carrier_curtailment_tables(network: pypsa.Network) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Return available, dispatched, and curtailed energy by carrier."""
    available = carrier_available_table(network)
    dispatched = carrier_dispatch_table(network)
    curtailed = (available - dispatched).clip(lower=0.0)
    return available, dispatched, curtailed


def co2_emissions_total(network: pypsa.Network) -> float:
    """Estimate total CO2 emissions if carrier emission factors are available."""
    generators = _as_dataframe(getattr(network, "generators", None))
    carriers = _as_dataframe(getattr(network, "carriers", None))
    dispatch = getattr(network.generators_t, "p", pd.DataFrame())
    if generators.empty or dispatch.empty or carriers.empty or "co2_emissions" not in carriers.columns:
        return 0.0
    carrier_factors = carriers["co2_emissions"].fillna(0.0)
    generator_factors = generators["carrier"].map(carrier_factors).fillna(0.0)
    efficiency = generators["efficiency"].fillna(1.0) if "efficiency" in generators.columns else pd.Series(1.0, index=generators.index)
    weights = _snapshot_weights(network)
    weighted_dispatch = dispatch.reindex(columns=generators.index, fill_value=0.0).mul(weights, axis=0)
    return float(weighted_dispatch.multiply(generator_factors.divide(efficiency), axis=1).sum().sum())


def line_loading_summary(network: pypsa.Network) -> dict[str, float]:
    """Return common line-loading statistics for a solved network."""
    loading = _line_loading_table(network)
    if loading.empty:
        return {
            "average_line_loading_percent": 0.0,
            "maximum_line_loading_percent": 0.0,
            "lines_above_60_avg": 0,
            "lines_above_70_avg": 0,
        }
    line_average = loading.mean(axis=0)
    return {
        "average_line_loading_percent": float(np.nanmean(loading.to_numpy(dtype=float))) if loading.size else 0.0,
        "maximum_line_loading_percent": float(np.nanmax(loading.to_numpy(dtype=float))) if loading.size else 0.0,
        "lines_above_60_avg": int((line_average > 60.0).sum()),
        "lines_above_70_avg": int((line_average > 70.0).sum()),
    }


def _bus_price_table(network: pypsa.Network) -> pd.DataFrame:
    """Return marginal price by bus and snapshot if available."""
    buses_t = getattr(network, "buses_t", None)
    if buses_t is None:
        return pd.DataFrame()
    if hasattr(buses_t, "marginal_price") and not getattr(buses_t.marginal_price, "empty", True):
        return buses_t.marginal_price
    return pd.DataFrame()


def parse_run_name(run_name: str) -> pd.Series:
    """Parse a run name into scenario metadata."""
    metadata = _network_label(run_name)
    return pd.Series({"run_name": run_name, **metadata})


def register_solved_networks(network_inputs: dict[str, str | pypsa.Network]) -> dict[str, pypsa.Network]:
    """Public wrapper that loads solved networks and annotates them with metadata."""
    return _resolve_networks(network_inputs)

### 2. Metric functions

Run-level, line-level, bus-level, and carrier-level metrics built from a solved PyPSA network.

In [150]:
# --- Metric configuration -------------------------------------------------

CONGESTION_THRESHOLD = 65.0  # Threshold for congestion analysis (% loading)
BUS_PRICE_THRESHOLD = 100.0
CURTAILMENT_CARRIERS = ("onwind", "offwind-ac", "solar")
WIND_CONDITIONS = ("windy", "notwindy", "windvariability")

# Adapt this mapping if your run names encode transmission cases differently.
TRANSMISSION_CASE_ALIASES = {
    "fixed": "fixed",
    "projects": "projects",
    "extendable": "extendable",
    "base2": "fixed",
    "baseTP": "projects",
    "scenario1": "fixed",
    "scenario2": "fixed",
    "scenario-ext": "extendable",
}

### 3. Common bottlenecks and transmission comparison

Line intersections across wind conditions and persistence checks across transmission cases.

In [151]:
def infer_transmission_case(run_name: str) -> str:
    """Map a run name to a normalized transmission case label."""

    lower_name = run_name.lower()
    for alias, normalized_case in TRANSMISSION_CASE_ALIASES.items():
        if alias.lower() in lower_name:
            return normalized_case
    return _network_label(run_name)["transmission_setting"]


def build_common_bottlenecks_table(
    line_stats_df: pd.DataFrame,
    scenario: str,
    transmission_case: str,
    threshold: float = CONGESTION_THRESHOLD,
) -> pd.DataFrame:
    """Return lines that exceed the average-loading threshold in all wind conditions.

    The input line table must include at least run_name, scenario, wind_condition,
    transmission_setting, line_id, bus0, bus1, average_loading_percent,
    maximum_loading_percent, and snapshots_above_60_percent.
    """

    subset = line_stats_df.copy()
    if subset.empty:
        return pd.DataFrame()

    if "transmission_case" not in subset.columns:
        subset["transmission_case"] = subset["run_name"].map(infer_transmission_case)
    subset["scenario"] = subset["scenario"].astype(str)
    subset["transmission_case"] = subset["transmission_case"].astype(str)

    subset = subset[(subset["scenario"] == scenario) & (subset["transmission_case"] == transmission_case)]
    if subset.empty:
        return pd.DataFrame()

    wind_tables = []
    threshold_sets = []
    for wind_condition in WIND_CONDITIONS:
        wind_subset = subset[subset["wind_condition"] == wind_condition].copy()
        if wind_subset.empty:
            continue
        wind_subset = wind_subset.set_index("line_id")
        threshold_sets.append(set(wind_subset.index[wind_subset["average_loading_percent"] > threshold]))
        wind_tables.append(
            wind_subset[
                [
                    "bus0",
                    "bus1",
                    "average_loading_percent",
                    "maximum_loading_percent",
                    "snapshots_above_60_percent",
                ]
            ].rename(
                columns={
                    "average_loading_percent": f"average_loading_percent_{wind_condition}",
                    "maximum_loading_percent": f"maximum_loading_percent_{wind_condition}",
                    "snapshots_above_60_percent": f"hours_above_threshold_{wind_condition}",
                }
            )
        )

    if len(wind_tables) < len(WIND_CONDITIONS):
        return pd.DataFrame()

    common_line_ids = set.intersection(*threshold_sets) if threshold_sets else set()
    if not common_line_ids:
        return pd.DataFrame()

    common_line_index = sorted(common_line_ids)
    common_lines = wind_tables[0].loc[common_line_index, ["bus0", "bus1"]].copy()
    for table in wind_tables:
        common_lines = common_lines.join(table.drop(columns=["bus0", "bus1"], errors="ignore"), how="left")

    common_lines.index.name = "line_id"
    common_lines = common_lines.reset_index()
    return common_lines.sort_values("line_id").reset_index(drop=True)


def compare_common_bottlenecks_transmission_cases(
    line_stats_df: pd.DataFrame,
    common_bottlenecks_df: pd.DataFrame,
    scenario: str,
    wind_condition: str,
    transmission_cases: tuple[str, ...] = ("fixed", "projects", "extendable"),
    threshold: float = CONGESTION_THRESHOLD,
) -> pd.DataFrame:
    """Compare the same bottleneck line ids across transmission cases for one wind condition."""

    if common_bottlenecks_df.empty or line_stats_df.empty:
        return pd.DataFrame()

    if "transmission_case" not in line_stats_df.columns:
        line_stats_df = line_stats_df.copy()
        line_stats_df["transmission_case"] = line_stats_df["run_name"].map(infer_transmission_case)

    reference_lines = common_bottlenecks_df["line_id"].dropna().astype(str).unique()
    subset = line_stats_df[
        (line_stats_df["scenario"].astype(str) == scenario)
        & (line_stats_df["wind_condition"].astype(str) == wind_condition)
        & (line_stats_df["line_id"].astype(str).isin(reference_lines))
        & (line_stats_df["transmission_case"].astype(str).isin(transmission_cases))
    ].copy()
    if subset.empty:
        return pd.DataFrame()

    base = common_bottlenecks_df.set_index("line_id")[["bus0", "bus1"]].copy()
    result = base

    for case in transmission_cases:
        case_subset = subset[subset["transmission_case"] == case].set_index("line_id")
        if case_subset.empty:
            continue
        case_columns = case_subset[
            ["average_loading_percent", "maximum_loading_percent", "snapshots_above_60_percent"]
        ].rename(
            columns={
                "average_loading_percent": f"average_loading_percent_{case}",
                "maximum_loading_percent": f"maximum_loading_percent_{case}",
                "snapshots_above_60_percent": f"hours_above_threshold_{case}",
            }
        )
        result = result.join(case_columns, how="left")

    for case in transmission_cases:
        average_column = f"average_loading_percent_{case}"
        if average_column in result.columns:
            result[f"still_above_threshold_{case}"] = result[average_column] > threshold

    if "average_loading_percent_fixed" in result.columns:
        for case in transmission_cases:
            average_column = f"average_loading_percent_{case}"
            if average_column in result.columns:
                result[f"change_vs_fixed_{case}"] = result[average_column] - result["average_loading_percent_fixed"]

    result.index.name = "line_id"
    return result.reset_index()

### 4. Aggregation tables

Load the solved networks, normalize run metadata, and build the combined tables used by the report.

In [ ]:
comparison = compare_common_bottlenecks_transmission_cases(
    line_stats_df,
    group[["line_id", "bus0", "bus1"]],
    scenario,
    wind_condition,
    ("fixed", "projects", "extendable"),
    CONGESTION_THRESHOLD
)

### 5. Report tables and CSV exports

Create the final tables used in the report and write them to CSV.

In [153]:
def export_table(table: pd.DataFrame, filename: str) -> Path:
    """Write a report table to CSV inside the report output directory."""

    output_path = report_output_dir / filename
    table.to_csv(output_path, index=False)
    return output_path


# Table A: run summary

### 6. Optional plotting helpers

Plotting functions stay separate from the data-processing and export steps.

In [154]:
from matplotlib.collections import LineCollection


def plot_line_loading_heatmap(line_stats_df: pd.DataFrame, top_n_lines: int = 30, run_order: list[str] | None = None, ax=None):
    """Plot a heatmap of average line loading by run for the most congested lines."""

    if line_stats_df.empty:
        raise ValueError("line_stats_df is empty")

    average_loading = (
        line_stats_df.groupby(["run_name", "line_id"], as_index=False)["average_loading_percent"].mean()
    )
    top_line_ids = (
        average_loading.groupby("line_id")["average_loading_percent"].max().sort_values(ascending=False).head(top_n_lines).index
    )
    heatmap_df = (
        average_loading[average_loading["line_id"].isin(top_line_ids)]
        .pivot(index="line_id", columns="run_name", values="average_loading_percent")
        .fillna(0.0)
    )
    if run_order is None:
        run_order = list(heatmap_df.columns)
    heatmap_df = heatmap_df.reindex(columns=run_order)

    if ax is None:
        _, ax = plt.subplots(figsize=(max(10, len(run_order) * 0.7), max(8, len(heatmap_df) * 0.25)))

    image = ax.imshow(heatmap_df.to_numpy(), aspect="auto", cmap="magma")
    ax.set_xticks(range(len(heatmap_df.columns)))
    ax.set_xticklabels(heatmap_df.columns, rotation=45, ha="right")
    ax.set_yticks(range(len(heatmap_df.index)))
    ax.set_yticklabels(heatmap_df.index)
    ax.set_title("Average line loading by run")
    ax.set_xlabel("Run")
    ax.set_ylabel("Line id")
    plt.colorbar(image, ax=ax, label="Average loading [%]")
    return ax


def plot_curtailment_by_carrier(curtailment_df: pd.DataFrame, ax=None):
    """Plot curtailed energy by carrier and run."""

    if curtailment_df.empty:
        raise ValueError("curtailment_df is empty")

    plot_df = (
        curtailment_df.groupby(["run_name", "carrier"], as_index=False)["curtailed_mwh"].sum()
        .pivot(index="run_name", columns="carrier", values="curtailed_mwh")
        .fillna(0.0)
    )

    if ax is None:
        _, ax = plt.subplots(figsize=(max(10, len(plot_df) * 0.8), 6))

    plot_df.plot(kind="bar", ax=ax, stacked=True, width=0.85)
    ax.set_title("Total curtailment by carrier and run")
    ax.set_xlabel("Run")
    ax.set_ylabel("Curtailment [MWh]")
    ax.legend(title="Carrier", bbox_to_anchor=(1.02, 1), loc="upper left")
    return ax


def build_bus_price_timeseries_df(networks: dict[str, pypsa.Network]) -> pd.DataFrame:
    """Return a long-form marginal-price table for boxplots and diagnostics."""

    rows = []
    for run_name, network in networks.items():
        price_table = _bus_price_table(network)
        if price_table.empty:
            continue
        metadata = {
            **parse_run_name(run_name).to_dict(),
            "transmission_case": infer_transmission_case(run_name),
        }
        long_table = price_table.reset_index(names="snapshot").melt(
            id_vars="snapshot",
            var_name="bus_id",
            value_name="marginal_price",
        )
        for key, value in metadata.items():
            long_table[key] = value
        rows.append(long_table)
    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()


def plot_marginal_price_boxplot(price_timeseries_df: pd.DataFrame, ax=None):
    """Plot marginal-price distributions by scenario and wind condition."""

    if price_timeseries_df.empty:
        raise ValueError("price_timeseries_df is empty")

    plot_df = price_timeseries_df.copy()
    plot_df["group_label"] = plot_df["scenario"].astype(str) + " | " + plot_df["wind_condition"].astype(str)
    grouped_labels = list(dict.fromkeys(plot_df["group_label"]))
    grouped_data = [plot_df.loc[plot_df["group_label"] == label, "marginal_price"].dropna().to_numpy() for label in grouped_labels]

    if ax is None:
        _, ax = plt.subplots(figsize=(max(10, len(grouped_labels) * 0.8), 6))

    ax.boxplot(grouped_data, labels=grouped_labels, showfliers=False)
    ax.set_title("Marginal-price distribution by scenario and wind condition")
    ax.set_xlabel("Scenario | wind condition")
    ax.set_ylabel("Marginal price [€/MWh]")
    ax.tick_params(axis="x", rotation=45)
    return ax


def plot_top_congested_lines_map(network: pypsa.Network, line_report_df: pd.DataFrame, run_name: str, ax=None):
    """Plot a pseudo-map of the top congested lines if coordinates are available."""

    if line_report_df.empty:
        raise ValueError("line_report_df is empty")

    if ax is None:
        _, ax = plt.subplots(figsize=(8, 8))

    lines = _as_dataframe(getattr(network, "lines", None))
    buses = _as_dataframe(getattr(network, "buses", None))
    if lines.empty or buses.empty or not {"x", "y"}.issubset(buses.columns):
        ax.text(0.5, 0.5, "No coordinates available", ha="center", va="center")
        ax.set_axis_off()
        return ax

    run_lines = line_report_df[line_report_df["run_name"] == run_name]
    if run_lines.empty:
        ax.text(0.5, 0.5, "No matching lines for run", ha="center", va="center")
        ax.set_axis_off()
        return ax

    line_ids = run_lines["line_id"].astype(str)
    available_line_ids = lines.index.astype(str)
    selected_line_ids = line_ids[line_ids.isin(available_line_ids)]
    if selected_line_ids.empty:
        ax.text(0.5, 0.5, "No matching line coordinates", ha="center", va="center")
        ax.set_axis_off()
        return ax

    selected = run_lines.set_index("line_id").loc[selected_line_ids]
    line_segments = []
    line_colors = []
    for line_id, row in selected.iterrows():
        if line_id not in lines.index:
            continue
        line = lines.loc[line_id]
        if pd.isna(line.get("bus0")) or pd.isna(line.get("bus1")):
            continue
        if line["bus0"] not in buses.index or line["bus1"] not in buses.index:
            continue
        start = (buses.loc[line["bus0"], "x"], buses.loc[line["bus0"], "y"])
        end = (buses.loc[line["bus1"], "x"], buses.loc[line["bus1"], "y"])
        line_segments.append([start, end])
        line_colors.append(float(row.get("average_loading_percent", 0.0)))

    if not line_segments:
        ax.text(0.5, 0.5, "No drawable line segments", ha="center", va="center")
        ax.set_axis_off()
        return ax

    collection = LineCollection(line_segments, cmap="inferno", linewidths=2.5)
    collection.set_array(np.asarray(line_colors, dtype=float))
    ax.add_collection(collection)
    ax.scatter(buses["x"], buses["y"], s=8, c="black", alpha=0.5)
    ax.autoscale()
    ax.set_title(f"Top congested lines: {run_name}")
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    plt.colorbar(collection, ax=ax, label="Average loading [%]")
    return ax


In [155]:
# === Persistent Congestion Analysis ===
# Find lines that are congested (>CONGESTION_THRESHOLD% loading) in at least 2 different fixed networks
# and rank them by how many scenarios they appear as congested

# Filter for fixed transmission case networks
fixed_line_stats = line_stats_df[line_stats_df['transmission_case'] == 'fixed'].copy()

# Mark lines as congested in each network
fixed_line_stats['is_congested'] = fixed_line_stats['average_loading_percent'] >= CONGESTION_THRESHOLD

# Group by line_id to find persistence
congestion_analysis = fixed_line_stats.groupby('line_id').agg({
    'bus0': 'first',
    'bus1': 'first',
    'carrier': 'first',
    'is_congested': 'sum',  # Count how many networks this line is congested in
    'run_name': 'count',  # Total appearances
    'average_loading_percent': ['min', 'max', 'mean']
}).reset_index()

# Flatten column names
congestion_analysis.columns = ['line_id', 'bus0', 'bus1', 'carrier', 'congestion_count', 
                               'total_networks', 'min_loading', 'max_loading', 'avg_loading']

# Filter for lines congested in at least 2 networks
persistent_congestion = congestion_analysis[congestion_analysis['congestion_count'] >= 2].copy()

# Add detailed network list for each congested line
congestion_networks = {}
for line_id in persistent_congestion['line_id'].values:
    congested_runs = fixed_line_stats[(fixed_line_stats['line_id'] == line_id) & 
                                       (fixed_line_stats['is_congested'])]['run_name'].tolist()
    congestion_networks[line_id] = ', '.join(sorted(congested_runs))

persistent_congestion['congested_in_networks'] = persistent_congestion['line_id'].map(congestion_networks)

# Sort by congestion count (descending), then by average loading
persistent_congestion = persistent_congestion.sort_values(
    by=['congestion_count', 'avg_loading'], 
    ascending=[False, False]
)

# Select and reorder columns for output
output_cols = ['line_id', 'bus0', 'bus1', 'carrier', 'congestion_count', 'total_networks',
               'min_loading', 'max_loading', 'avg_loading', 'congested_in_networks']
persistent_congestion_output = persistent_congestion[output_cols].copy()

# Round numeric columns
numeric_cols = ['min_loading', 'max_loading', 'avg_loading']
persistent_congestion_output[numeric_cols] = persistent_congestion_output[numeric_cols].round(2)

# Export to CSV
output_path = Path(report_output_dir) / 'persistent_congestion_lines.csv'
persistent_congestion_output.to_csv(output_path, index=False)

print(f"\n=== Persistent Congestion Analysis (Fixed Networks, Threshold >= {CONGESTION_THRESHOLD}%) ===")
print(f"Lines congested in >= 2 networks: {len(persistent_congestion_output)}")
print(f"\nExported to: {output_path}")
print(f"\nTop 20 most persistent congested lines:")
print(persistent_congestion_output.head(20).to_string(index=False))


=== Persistent Congestion Analysis (Fixed Networks, Threshold >= 65.0%) ===
Lines congested in >= 2 networks: 31

Exported to: /home/lucakristin/Desktop/my_pypsa/pypsa-eur/comparison/Comparison2/persistent_congestion_lines.csv

Top 20 most persistent congested lines:
line_id    bus0    bus1 carrier  congestion_count  total_networks  min_loading  max_loading  avg_loading                                                 congested_in_networks
    622  DE0 61  DE0 96      AC                 3               9        26.50        70.00        49.10 germany_base2_windy, germany_scenario1_windy, germany_scenario2_windy
     99  DE0 13  DE0 69      AC                 2               9        50.62        65.61        59.05                      germany_scenario1_windy, germany_scenario2_windy
     60 DE0 115  DE0 46      AC                 2               9        25.11        69.65        51.02                      germany_scenario1_windy, germany_scenario2_windy
    119 DE0 136 DE0 317      AC

In [156]:
# === Diagnostic: Congestion Persistence Analysis ===
# Check how many wind conditions each line is congested in
# This will reveal if there are any lines congested in 2+ conditions (even if not all 3)

print("=== Congestion Persistence Across Wind Conditions ===\n")

for scenario in line_stats_df['scenario'].unique():
    for transmission_case in line_stats_df['transmission_case'].unique():
        subset = line_stats_df[
            (line_stats_df['scenario'] == scenario) & 
            (line_stats_df['transmission_case'] == transmission_case)
        ].copy()
        
        if subset.empty:
            continue
        
        # Count how many wind conditions each line is congested in
        congestion_counts = subset.groupby('line_id')['average_loading_percent'].apply(
            lambda x: (x > CONGESTION_THRESHOLD).sum()
        )
        
        congestion_dist = congestion_counts.value_counts().sort_index(ascending=False)
        
        print(f"{scenario} | {transmission_case}:")
        print(f"  Lines congested in 3 wind conditions: {(congestion_counts == 3).sum()}")
        print(f"  Lines congested in 2 wind conditions: {(congestion_counts == 2).sum()}")
        print(f"  Lines congested in 1 wind condition:  {(congestion_counts == 1).sum()}")
        print(f"  (Using threshold: {CONGESTION_THRESHOLD}% loading)")
        
        # Show top lines congested in 2+ conditions
        top_persistent = congestion_counts[congestion_counts >= 2].sort_values(ascending=False).head(10)
        if len(top_persistent) > 0:
            print(f"  Top persistent lines (2+ conditions):")
            for line_id, count in top_persistent.items():
                line_data = subset[subset['line_id'] == line_id]
                loadings = line_data[['wind_condition', 'average_loading_percent']].set_index('wind_condition')['average_loading_percent'].to_dict()
                print(f"    {line_id}: {count} conditions - {loadings}")
        print()

=== Congestion Persistence Across Wind Conditions ===

germany_base2 | fixed:
  Lines congested in 3 wind conditions: 0
  Lines congested in 2 wind conditions: 0
  Lines congested in 1 wind condition:  2
  (Using threshold: 65.0% loading)

germany_baseTP | projects:
  Lines congested in 3 wind conditions: 0
  Lines congested in 2 wind conditions: 1
  Lines congested in 1 wind condition:  1
  (Using threshold: 65.0% loading)
  Top persistent lines (2+ conditions):
    127: 2 conditions - {'notwindy': 52.193285978415645, 'windvariability': 68.80647876736167, 'windy': 68.80644014342762}

germany_scenario-ext | extendable:
  Lines congested in 3 wind conditions: 0
  Lines congested in 2 wind conditions: 0
  Lines congested in 1 wind condition:  8
  (Using threshold: 65.0% loading)

germany_scenario1 | fixed:
  Lines congested in 3 wind conditions: 0
  Lines congested in 2 wind conditions: 0
  Lines congested in 1 wind condition:  31
  (Using threshold: 65.0% loading)

germany_scenario2 | f

In [157]:
# === Quick diagnostic: Check max line loadings ===
print("=== Line Loading Statistics ===\n")
print(f"Max line loading across all lines/scenarios: {line_stats_df['average_loading_percent'].max():.2f}%")
print(f"Min line loading: {line_stats_df['average_loading_percent'].min():.2f}%")
print(f"Mean line loading: {line_stats_df['average_loading_percent'].mean():.2f}%")
print(f"Median line loading: {line_stats_df['average_loading_percent'].median():.2f}%")
print()

# Show top 20 most loaded lines
top_20 = line_stats_df.nlargest(20, 'average_loading_percent')[['run_name', 'line_id', 'bus0', 'bus1', 'average_loading_percent']].drop_duplicates('line_id')
print(f"Top 20 most loaded lines:")
print(top_20.to_string(index=False))

=== Line Loading Statistics ===

Max line loading across all lines/scenarios: 70.00%
Min line loading: 0.27%
Mean line loading: 19.52%
Median line loading: 15.29%

Top 20 most loaded lines:
               run_name line_id    bus0    bus1  average_loading_percent
germany_scenario1_windy     622  DE0 61  DE0 96                69.999971
germany_scenario1_windy     119 DE0 136 DE0 317                69.999922
germany_scenario1_windy      94 DE0 128 DE0 174                69.999650
germany_scenario1_windy      47 DE0 112 DE0 349                69.998860
germany_scenario1_windy     569 DE0 372  DE0 44                69.993942
germany_scenario1_windy     178  DE0 16 DE0 304                69.972493
germany_scenario1_windy     487  DE0 30  DE0 77                69.802956
germany_scenario1_windy      60 DE0 115  DE0 46                69.645491
germany_scenario1_windy     383 DE0 246 DE0 267                69.448341
germany_scenario1_windy     103 DE0 131  DE0 33                69.374064
